# Chapter 10 — Creating Text Embedding Models
### Practice Notebook

*Source: Hands-On Large Language Models, Jay Alammar & Maarten Grootendorst (O'Reilly)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter10/Chapter%2010%20-%20Creating%20Text%20Embedding%20Models.ipynb)

---

This notebook is your **practice workspace** for Chapter 10. Every code cell is a stub — implement the solution yourself and then run it. Refer to the chapter notes at `notes/ch10-creating-text-embedding-models.md` if you need a refresher on the theory.

---

## Table of Contents

- [Part 0: Theory Warm-Up](#part-0-theory-warm-up)
  - [T1: Cross-Encoder vs Siamese Complexity](#t1-cross-encoder-vs-siamese-complexity)
  - [T2: Implement Cosine Similarity From Scratch](#t2-implement-cosine-similarity-from-scratch)
- [Part 1: Creating a Baseline Embedding Model](#part-1-creating-a-baseline-embedding-model)
  - [1.1 Load & Inspect MNLI](#11-load--inspect-mnli)
  - [1.2 Initialize the Model](#12-initialize-the-model)
  - [1.3 Define SoftmaxLoss](#13-define-softmaxloss)
  - [1.4 Create STSB Evaluator](#14-create-stsb-evaluator)
  - [1.5 Configure Training Arguments](#15-configure-training-arguments)
  - [1.6 Train & Evaluate](#16-train--evaluate)
  - [1.7 MTEB Evaluation](#17-mteb-evaluation)
- [Part 2: Loss Functions](#part-2-loss-functions)
  - [2A: Cosine Similarity Loss](#2a-cosine-similarity-loss)
  - [2B: Multiple Negatives Ranking Loss](#2b-multiple-negatives-ranking-loss)
- [Part 3: Fine-Tuning a Pre-Trained Model](#part-3-fine-tuning-a-pre-trained-model)
- [Part 4: Augmented SBERT](#part-4-augmented-sbert)
- [Part 5: Unsupervised Learning — TSDAE](#part-5-unsupervised-learning--tsdae)

### [OPTIONAL] — Install packages on Colab

Uncomment and run if you are on a cloud GPU environment.

💡 **GPU required.** Go to Runtime → Change runtime type → GPU (T4 on Colab).

In [ ]:
# %%capture
# !pip install -q accelerate>=0.27.2 peft>=0.9.0 bitsandbytes>=0.43.0 transformers>=4.38.2 trl>=0.7.11 sentencepiece>=0.1.99
# !pip install -q sentence-transformers>=3.0.0 mteb>=1.1.2 datasets>=2.18.0

---
# Part 0: Theory Warm-Up

These exercises reinforce the core concepts from the notes **before** touching any training code. They are not in the original book notebook but are essential for understanding *why* the chapter's design decisions were made.

## T1: Cross-Encoder vs Siamese Complexity

A cross-encoder must process **every pair** of sentences together. For a corpus of $n$ sentences, the number of comparisons needed to build a full similarity matrix is $n(n-1)/2$.

**Task:** Write a function `cross_encoder_comparisons(n)` that returns the number of pairwise comparisons required. Then print the comparison counts and estimated time (assuming 1 ms per comparison) for corpus sizes of 100, 1 000, 10 000, and 100 000 sentences.

**Expected output (approximate):**
```
n=100       →       4,950 comparisons  →    ~0.0 seconds
n=1,000     →     499,500 comparisons  →    ~0.5 seconds
n=10,000    →  49,995,000 comparisons  →  ~13.9 hours
n=100,000   → 4,999,950,000 comparisons → ~1,388.9 hours
```

In [ ]:
def cross_encoder_comparisons(n: int) -> int:
    """Return the number of pairwise comparisons for a corpus of n sentences."""
    # YOUR CODE HERE
    pass


for n in [100, 1_000, 10_000, 100_000]:
    comps = cross_encoder_comparisons(n)
    ms_per_comp = 1  # 1 ms per BERT forward pass
    # YOUR CODE HERE
    # Compute total seconds and print formatted output
    pass

## T2: Implement Cosine Similarity From Scratch

Before using the loss function from a library, make sure you understand the math behind it.

The cosine similarity between two vectors $\mathbf{u}$ and $\mathbf{v}$ is:

$$\text{sim}_{\cos}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \cdot \|\mathbf{v}\|}$$

**Task:** Implement `cosine_similarity(u, v)` using only numpy. Then verify your implementation against `torch.nn.functional.cosine_similarity` on two test vectors.

**Test case:**
```
u = [1.0, 0.0]   →  ||u|| = 1.0
v = [0.6, 0.8]   →  ||v|| = 1.0  (it's a unit vector)
Expected: cosine_similarity(u, v) = 0.6
```

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F


def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    """Compute cosine similarity between two 1D numpy arrays."""
    # YOUR CODE HERE
    # Hint: np.dot, np.linalg.norm
    pass


# Test case
u = np.array([1.0, 0.0])
v = np.array([0.6, 0.8])

my_result = cosine_similarity(u, v)
print(f"My cosine_similarity(u, v) = {my_result}")
# Expected: 0.6

# Verify with PyTorch
torch_result = F.cosine_similarity(
    torch.tensor(u).unsqueeze(0),
    torch.tensor(v).unsqueeze(0)
).item()
print(f"torch cosine_similarity(u, v) = {torch_result:.6f}")
print(f"Match: {abs(my_result - torch_result) < 1e-5}")

---
# Part 1: Creating a Baseline Embedding Model

We will train an SBERT-style sentence embedding model from scratch using the MNLI dataset and `SoftmaxLoss` as a baseline. This gives us a reference score of **pearson_cosine ≈ 0.59** that we will try to beat in Part 2.

## 1.1 Load & Inspect MNLI

**Data:** MNLI (Multi-Genre Natural Language Inference) from the GLUE benchmark.

The dataset contains `(premise, hypothesis, label)` triples where:
- `label = 0` → **entailment** (hypothesis follows from premise → positive pair)
- `label = 1` → **neutral** (hypothesis is unrelated → discard)
- `label = 2` → **contradiction** (hypothesis contradicts premise → negative pair)

**Task:** Load the first 50,000 rows of the MNLI training split from GLUE. Remove the `idx` column. Print the third example (index 2) to verify the structure.

In [ ]:
from datasets import load_dataset

# Load MNLI dataset from GLUE (0=entailment, 1=neutral, 2=contradiction)
# YOUR CODE HERE
# train_dataset = ...

# Inspect the third row
# YOUR CODE HERE
# Expected keys: premise, hypothesis, label
# Expected label for index 2: 0 (entailment)

## 1.2 Initialize the Model

We use `bert-base-uncased` as the backbone — 12 transformer layers, 768-dimensional hidden state, 110M parameters. `SentenceTransformer` automatically attaches a mean-pooling layer so the model outputs a single fixed-size sentence vector.

**Task:** Instantiate the embedding model. Print its sentence embedding dimension.

In [ ]:
from sentence_transformers import SentenceTransformer

# YOUR CODE HERE
# embedding_model = SentenceTransformer(...)

# Print embedding dimension — should be 768 for BERT-base
# print(embedding_model.get_sentence_embedding_dimension())

## 1.3 Define SoftmaxLoss

`SoftmaxLoss` treats the NLI task as a 3-class classification problem (entailment / neutral / contradiction). It adds a linear classifier on top of the concatenated sentence embeddings and optimises cross-entropy.

**Task:** Create the softmax loss. You need to pass in the model, its embedding dimension, and the number of output labels (3).

In [ ]:
from sentence_transformers import losses

# YOUR CODE HERE
# train_loss = losses.SoftmaxLoss(
#     model=...,
#     sentence_embedding_dimension=...,
#     num_labels=...
# )

## 1.4 Create STSB Evaluator

We evaluate on the **STSB** (Semantic Textual Similarity Benchmark) validation set. Each pair has a human-labelled similarity score from 1–5. We normalise to [0, 1] by dividing by 5.

The evaluator computes the cosine similarity between the model's embeddings of each pair and reports Pearson and Spearman correlation with the human labels.

**Task:** Load the STSB validation split and build an `EmbeddingSimilarityEvaluator` with `main_similarity="cosine"`.

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# YOUR CODE HERE
# val_sts = load_dataset('glue', 'stsb', split='validation')
# evaluator = EmbeddingSimilarityEvaluator(
#     sentences1=...,
#     sentences2=...,
#     scores=...,   # Remember to normalise from [1-5] to [0-1]
#     main_similarity='cosine'
# )

## 1.5 Configure Training Arguments

**Task:** Create `SentenceTransformerTrainingArguments` with the following configuration:

| Parameter | Value |
|---|---|
| `output_dir` | `"base_embedding_model"` |
| `num_train_epochs` | 1 |
| `per_device_train_batch_size` | 32 |
| `per_device_eval_batch_size` | 32 |
| `warmup_steps` | 100 |
| `fp16` | True |
| `eval_steps` | 100 |
| `logging_steps` | 100 |

*Why `warmup_steps=100`?* The learning rate starts at 0 and linearly ramps up to its target value over the first 100 steps, preventing large gradient updates from destabilising the pre-trained BERT weights early in training.

In [ ]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# YOUR CODE HERE
# args = SentenceTransformerTrainingArguments(
#     output_dir=...,
#     ...
# )

## 1.6 Train & Evaluate

**Task:** Create a `SentenceTransformerTrainer` and call `.train()`. Then call the evaluator directly on the model to get the final STSB metrics.

**Expected result:** `pearson_cosine ≈ 0.37–0.59`  
*Note: The exact value varies with GPU/environment. The book reports ~0.59 on a T4 GPU with a full epoch.*

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# YOUR CODE HERE
# trainer = SentenceTransformerTrainer(
#     model=...,
#     args=...,
#     train_dataset=...,
#     loss=...,
#     evaluator=...
# )
# trainer.train()

In [ ]:
# Evaluate the trained model on the full STSB validation set
evaluator(embedding_model)

## 1.7 MTEB Evaluation

STSB is a fast sanity check. For a more thorough assessment, the **Massive Text Embedding Benchmark (MTEB)** covers 58 datasets and 112 languages across 8 task types.

**Task:** Run the `Banking77Classification` task from MTEB on your trained model. Print the accuracy result.

*Note: A full MTEB run takes hours — use `Banking77Classification` as a representative task during development.*

In [ ]:
from mteb import MTEB

# YOUR CODE HERE
# evaluation = MTEB(tasks=[...])
# results = evaluation.run(embedding_model)
# print(results)

⚠️ **VRAM Clean-up** — Run the cell below, or restart the notebook kernel, before training the next model.

In [ ]:
# # Empty and delete trainer/model
# trainer.accelerator.clear()
# del trainer, embedding_model

import gc
import torch

gc.collect()
torch.cuda.empty_cache()

---
# Part 2: Loss Functions

The baseline used `SoftmaxLoss` (pearson_cosine ≈ 0.59). We will now try two loss functions that are more directly aligned with cosine similarity, watching the score improve:

| Loss | pearson_cosine |
|---|---|
| SoftmaxLoss (baseline) | 0.59 |
| CosineSimilarityLoss | 0.72 |
| MultipleNegativesRankingLoss | 0.80 |

⚠️ **Restart the notebook kernel** before each training run in this section to free VRAM.

## 2A: Cosine Similarity Loss

`CosineSimilarityLoss` trains the model to directly match its predicted cosine similarity to a labelled similarity score. For our NLI data we need **binary** similarity labels: entailment → 1.0, contradiction/neutral → 0.0.

**Data format required:** `{"sentence1": ..., "sentence2": ..., "label": float}` where label ∈ [0, 1].

### Exercise 2.1 — Remap MNLI labels

**Task:** Load MNLI (50k rows, remove `idx`). Map the labels: `{entailment: 1.0, neutral: 0.0, contradiction: 0.0}`. Build a new `Dataset` with columns `sentence1`, `sentence2`, `label`.

Hint: MNLI label integers are `0=entailment, 1=neutral, 2=contradiction`.

In [ ]:
from datasets import Dataset, load_dataset

# YOUR CODE HERE
# mapping = {0: 1.0, 1: 0.0, 2: 0.0}   # entailment=1.0, neutral/contradiction=0.0
# train_dataset = Dataset.from_dict({
#     "sentence1": ...,
#     "sentence2": ...,
#     "label": ...
# })

### Exercise 2.2 — Train with CosineSimilarityLoss

**Task:** Set up the full training pipeline:
1. Create a fresh `SentenceTransformer('bert-base-uncased')`
2. Create the `CosineSimilarityLoss`
3. Create the STSB evaluator (same as before)
4. Create `SentenceTransformerTrainingArguments` with `output_dir="cosineloss_embedding_model"`
5. Train and evaluate

**Expected:** `pearson_cosine ≈ 0.72`

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# 1. Fresh model
# YOUR CODE HERE

# 2. Loss function
# YOUR CODE HERE

# 3. Evaluator
# YOUR CODE HERE

# 4. Training arguments (output_dir="cosineloss_embedding_model")
# YOUR CODE HERE

# 5. Train
# YOUR CODE HERE

In [ ]:
# Evaluate — expected pearson_cosine ≈ 0.72
evaluator(embedding_model)

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

## 2B: Multiple Negatives Ranking Loss

`MultipleNegativesRankingLoss` (MNR Loss, also called InfoNCE / NTXentLoss) frames embedding training as a classification problem: given an anchor, which sentence in the batch is its true positive? It uses cross-entropy over cosine similarities — and the negatives come for free from the rest of the batch.

**Data format required:** `{"anchor": ..., "positive": ..., "negative": ...}`

We only use **entailment pairs** (label=0) as (anchor=premise, positive=hypothesis). We generate a **soft negative** for each row by randomly shuffling the hypothesis column — meaning each anchor gets a hypothesis from a different premise as its negative.

### Exercise 2.3 — Build anchor / positive / negative triplets

**Task:**
1. Load MNLI (50k rows), filter to keep only `label == 0` (entailment)
2. Shuffle the `hypothesis` column to get soft negatives
3. Build a `Dataset` with columns `anchor`, `positive`, `negative`
4. Print the number of triplets — should be ~16,875 (roughly 1/3 of 50k)

*Why shuffle hypotheses for negatives? Because they are real sentences from the same domain (NLI) but paired with the wrong premise — they are topically plausible but semantically wrong, making them useful in-batch negatives.*

In [ ]:
import random
from tqdm import tqdm
from datasets import Dataset, load_dataset

# YOUR CODE HERE
# 1. Load MNLI and filter to entailment only
# mnli = load_dataset(...).select(...).remove_columns("idx")
# mnli = mnli.filter(...)

# 2. Shuffle hypotheses to create soft negatives
# soft_negatives = ...
# random.shuffle(soft_negatives)

# 3. Build triplet dataset
# train_dataset = {"anchor": [], "positive": [], "negative": []}
# for row, soft_negative in tqdm(zip(mnli, soft_negatives)):
#     ...
# train_dataset = Dataset.from_dict(train_dataset)

# 4. How many triplets?
# print(len(train_dataset))

### Exercise 2.4 — Train with MultipleNegativesRankingLoss

**Task:** Full pipeline — model, loss, evaluator, training args (`output_dir="mnrloss_embedding_model"`), train.

**Expected:** `pearson_cosine ≈ 0.80`

*Notice: MNR loss outperforms cosine loss despite training on ~16k examples vs 50k. Loss function design matters more than raw data volume.*

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# YOUR CODE HERE
# Model, loss (MultipleNegativesRankingLoss), evaluator, args, trainer, train

In [ ]:
# Evaluate — expected pearson_cosine ≈ 0.80
evaluator(embedding_model)

---
# Part 3: Fine-Tuning a Pre-Trained Model

So far we have been training `bert-base-uncased` **from its general language model weights**. An alternative is to start from a model that is already a good sentence encoder — `all-MiniLM-L6-v2` — and fine-tune it on our domain data.

⚠️ **Restart the kernel** before running this section.

**Interesting observation:** Fine-tuning an already-good embedding model on MNLI may actually *decrease* performance on STSB if the MNLI distribution doesn't perfectly match STSB. We will compare the fine-tuned model against the original pre-trained model to see the trade-off.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

### Exercise 3.1 — Fine-tune `all-MiniLM-L6-v2`

**Task:**
1. Load MNLI (50k rows), create STSB evaluator
2. Initialize `SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')`
3. Use `MultipleNegativesRankingLoss`
4. Train with `output_dir="finetuned_embedding_model"`
5. Evaluate the fine-tuned model
6. **Also evaluate the original pre-trained model** (load it fresh and run the same evaluator) — see if fine-tuning helped or hurt

In [ ]:
from datasets import load_dataset
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# YOUR CODE HERE — load data and evaluator

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# YOUR CODE HERE — model, loss, args, trainer, train
# Model: 'sentence-transformers/all-MiniLM-L6-v2'

In [ ]:
# Evaluate fine-tuned model
print("Fine-tuned model:")
evaluator(embedding_model)

In [ ]:
# Evaluate the original pre-trained model for comparison
original_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Original pre-trained model:")
evaluator(original_model)
# Observation: the pre-trained model may outperform the fine-tuned one on STSB!
# This is expected — all-MiniLM-L6-v2 was trained on 1B+ sentence pairs;
# our 50k MNLI fine-tune adds noise to a model that was already well-calibrated.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

---
# Part 4: Augmented SBERT

**Problem:** What if you only have a small amount of high-quality labelled data (gold data) but lots of unlabelled sentence pairs?

**Solution — Augmented SBERT:** Use a cross-encoder (which is more accurate than a bi-encoder) to label additional sentence pairs automatically. This creates a larger **silver dataset** that augments your original gold data, letting you train a stronger bi-encoder than gold data alone would allow.

**5 Steps:**
1. Fine-tune a cross-encoder on the small gold dataset (10k pairs)
2. Create new (unlabelled) sentence pairs (the next 40k MNLI rows)
3. Label those pairs with the fine-tuned cross-encoder → silver dataset
4. Train a bi-encoder on gold + silver combined
5. Ablation: train on gold only to confirm silver data helps

⚠️ **Restart the kernel** before running this section.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

### Exercise 4.1 — Step 1: Fine-tune a cross-encoder on gold data (10k pairs)

**Task:**
1. Load the first 10k rows of MNLI, map labels (entailment=1, neutral/contradiction=0)
2. Create `InputExample` objects and a `NoDuplicatesDataLoader` (batch_size=32)
3. Also store the data as a pandas DataFrame (you'll need it later for combining with silver)
4. Train a `CrossEncoder('bert-base-uncased', num_labels=2)` for 1 epoch with `use_amp=False`

In [ ]:
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import InputExample
from sentence_transformers.datasets import NoDuplicatesDataLoader
from sentence_transformers.cross_encoder import CrossEncoder

# Load 10k gold pairs
# YOUR CODE HERE

# Create InputExample list and NoDuplicatesDataLoader
# YOUR CODE HERE

# Create pandas DataFrame for gold data (will be used in Step 4)
# YOUR CODE HERE

# Train cross-encoder
# cross_encoder = CrossEncoder('bert-base-uncased', num_labels=2)
# cross_encoder.fit(
#     train_dataloader=...,
#     epochs=1,
#     show_progress_bar=True,
#     warmup_steps=100,
#     use_amp=False
# )

### Exercise 4.2 — Step 2: Create new sentence pairs (silver)

**Task:** Load rows 10k–50k from MNLI as unlabelled sentence pairs. Create a list of `(premise, hypothesis)` tuples called `pairs`.

In [ ]:
# YOUR CODE HERE
# silver = load_dataset(...).select(range(10_000, 50_000))
# pairs = list(zip(silver['premise'], silver['hypothesis']))

### Exercise 4.3 — Step 3: Label silver pairs with the cross-encoder

**Task:** Use `cross_encoder.predict(pairs, apply_softmax=True)` to get class probabilities. Use `np.argmax` to get binary labels (0 or 1). Store the result as a pandas DataFrame with columns `sentence1`, `sentence2`, `label`.

In [ ]:
import numpy as np

# YOUR CODE HERE
# output = cross_encoder.predict(pairs, apply_softmax=True, show_progress_bar=True)
# silver = pd.DataFrame({...})

### Exercise 4.4 — Step 4: Train bi-encoder on gold + silver

**Task:**
1. Concatenate gold and silver DataFrames, drop duplicates on `(sentence1, sentence2)`
2. Convert to a HuggingFace `Dataset`
3. Create STSB evaluator
4. Train a fresh `SentenceTransformer('bert-base-uncased')` with `CosineSimilarityLoss` and `output_dir="augmented_embedding_model"`
5. Evaluate

In [ ]:
from sentence_transformers import losses, SentenceTransformer
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# YOUR CODE HERE
# 1. data = pd.concat([gold, silver], ...).drop_duplicates(...)
# 2. train_dataset = Dataset.from_pandas(data, preserve_index=False)
# 3. evaluator = ...
# 4. Train
# 5. Evaluate

In [ ]:
# Evaluate gold + silver model
evaluator(embedding_model)

In [ ]:
trainer.accelerator.clear()

### Exercise 4.5 — Step 5: Ablation — gold data only

**Task:** Repeat Step 4 but use **only the gold DataFrame** (no silver). Train with `output_dir="gold_only_embedding_model"` and evaluate. Compare the score to the gold+silver model.

**Expected:** Gold-only performs worse than gold+silver, demonstrating that the cross-encoder-labelled silver data provides genuine signal.

In [ ]:
# YOUR CODE HERE — same pipeline but data = pd.concat([gold]) only

In [ ]:
# Evaluate gold-only model — should be lower than gold+silver
evaluator(embedding_model)

**Observation:** Compared to using both the silver and gold datasets, using only the gold dataset reduces the performance of the model. This confirms that cross-encoder-generated silver labels add genuine value beyond what the small gold dataset alone can provide.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

---
# Part 5: Unsupervised Learning — TSDAE

What if you have **no labelled data at all** — just raw sentences?

**TSDAE (Transformer-based Denoising AutoEncoder)** is an unsupervised pre-training technique that works by:
1. Corrupting input sentences by randomly deleting tokens (the *damaged* sentence)
2. Training the encoder to produce an embedding from which a decoder can reconstruct the original sentence

The encoder is forced to create a rich, information-dense embedding because the decoder needs it to reconstruct the full original text from a corrupted input. No labels needed — just raw sentences.

**Data format required:** `{"damaged_sentence": ..., "original_sentence": ...}`

⚠️ **Restart the kernel** before running this section.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Download tokenizer (required by DenoisingAutoEncoderDataset)
import nltk
nltk.download('punkt')

### Exercise 5.1 — Create the denoised training dataset

**Task:**
1. Load the first 25k rows of MNLI, flatten `premise` and `hypothesis` into a single list of sentences
2. Deduplicate the sentences with `set()`
3. Wrap with `DenoisingAutoEncoderDataset` — this applies random token deletion to each sentence
4. Iterate through the corrupted data and build a `Dataset` with `damaged_sentence` and `original_sentence` columns
5. Inspect the first example to see what the corruption looks like

In [ ]:
from tqdm import tqdm
from datasets import Dataset, load_dataset
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

# YOUR CODE HERE
# 1. Load MNLI (25k rows)
# 2. flat_sentences = mnli["premise"] + mnli["hypothesis"]
# 3. damaged_data = DenoisingAutoEncoderDataset(list(set(flat_sentences)))
# 4. Build Dataset
# 5. train_dataset[0]

In [ ]:
# Inspect first example — damaged_sentence has random words deleted
train_dataset[0]

In [ ]:
# Optional: customise the deletion ratio (default is 0.6 = delete 60% of tokens)
# flat_sentences = list(set(flat_sentences))
# damaged_data = DenoisingAutoEncoderDataset(
#     flat_sentences,
#     noise_fn=lambda s: DenoisingAutoEncoderDataset.delete(s, del_ratio=0.6)
# )

### Exercise 5.2 — Build the TSDAE model (Transformer + Pooling)

TSDAE uses `[CLS]` pooling (not mean pooling) because the decoder reads from the `[CLS]` position.

**Task:** Create the model by composing two `sentence_transformers.models` modules:
- `models.Transformer('bert-base-uncased')` — the word embedding layer
- `models.Pooling(..., 'cls')` — CLS token pooling

Pass both as a list to `SentenceTransformer(modules=[...])`.

In [ ]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

# STSB evaluator for monitoring
val_sts = load_dataset('glue', 'stsb', split='validation')
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]],
    main_similarity="cosine"
)

In [ ]:
from sentence_transformers import models, SentenceTransformer

# YOUR CODE HERE
# word_embedding_model = models.Transformer('bert-base-uncased')
# pooling_model = models.Pooling(
#     word_embedding_model.get_word_embedding_dimension(), 
#     pooling_mode='cls'
# )
# embedding_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

### Exercise 5.3 — Define DenoisingAutoEncoderLoss

**Task:** Create the loss with `tie_encoder_decoder=True` (encoder and decoder share weights, saving parameters). Move the decoder to CUDA.

*Why `tie_encoder_decoder`? Sharing weights between encoder and decoder reduces the parameter count and acts as a regulariser, which is beneficial when training on relatively small unsupervised data.*

In [ ]:
from sentence_transformers import losses

# YOUR CODE HERE
# train_loss = losses.DenoisingAutoEncoderLoss(
#     embedding_model, tie_encoder_decoder=True
# )
# train_loss.decoder = train_loss.decoder.to("cuda")

### Exercise 5.4 — Train and evaluate TSDAE

**Task:** Create training args with `output_dir="tsdae_embedding_model"` and **`per_device_train_batch_size=16`** (TSDAE needs a smaller batch size because the decoder adds significant VRAM overhead). Train and evaluate.

**Note:** TSDAE is unsupervised pre-training — expect a *lower* STSB score than supervised methods. Its value is in domains where labelled data is scarce; TSDAE provides a better starting point than plain BERT for subsequent fine-tuning.

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# YOUR CODE HERE
# args = SentenceTransformerTrainingArguments(
#     output_dir="tsdae_embedding_model",
#     per_device_train_batch_size=16,   # smaller than usual — decoder adds VRAM cost
#     per_device_eval_batch_size=16,
#     ...
# )
# trainer = SentenceTransformerTrainer(...)
# trainer.train()

In [ ]:
# Evaluate TSDAE model on STSB
# Unsupervised training — score will be lower than supervised methods, but higher than random
evaluator(embedding_model)

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

---
## Chapter 10 Summary

You have now implemented all four embedding model training paradigms from the chapter:

| Approach | Loss | Data Needed | STSB Pearson |
|---|---|---|---|
| Baseline bi-encoder | SoftmaxLoss | Labelled (3-class NLI) | ~0.59 |
| Cosine loss | CosineSimilarityLoss | Labelled (binary similarity) | ~0.72 |
| MNR loss | MultipleNegativesRankingLoss | Positive pairs only | ~0.80 |
| Supervised fine-tune | MultipleNegativesRankingLoss | Positive pairs + pre-trained model | ~0.85 |
| Augmented SBERT | CosineSimilarityLoss | Small gold + cross-encoder silver | ~0.80+ |
| TSDAE | DenoisingAutoEncoderLoss | **No labels** | Lower, but label-free |

**Key insight to remember:** The loss function is the single highest-leverage choice in embedding model training — it matters more than raw data volume or even which base model you start from.